# Bab 12: Variabel Laten Kontinu

Notebook pendamping buku *Pengenalan Pola dan Machine Learning: Pendekatan Probabilistik dengan Python*.

Jalankan sel dari atas ke bawah (**Runtime → Run all** di Colab). Sel pertama menyiapkan folder kode dan modul data sintetis; setiap sel berikutnya adalah satu listing dari buku, dengan nomor penanda `# (1)`, `# (2)`, … yang dijelaskan di bawah listing pada buku. Angka keluaran bisa sedikit berbeda dari buku bila versi pustakanya berbeda (lihat Lampiran A).

In [ ]:
# Menyiapkan folder kode/ dan modul pendukung bab ini
import os, pathlib
BERKAS = {
"bab12/data_provinsi.py": "\"\"\"Profil sosial-ekonomi 38 provinsi sintetis (ilustrasi).\n\nSetiap provinsi diberi kode wilayah (SUM, JAW, BNT, KAL, SUL, MAL, PAP) dan\nnomor urut, bukan nama asli, karena angkanya dibangkitkan dari dua faktor laten\n(\"pembangunan\" dan \"urbanisasi\") ditambah derau, bukan data resmi.\n\"\"\"\nimport numpy as np\n\nINDIKATOR = [\"miskin\", \"IPM\", \"RLS\", \"UHH\", \"pengangguran\", \"gini\",\n             \"air_layak\", \"internet\", \"PDRB_log\", \"perkotaan\", \"stunting\"]\nWILAYAH = {\"SUM\": 10, \"JAW\": 6, \"BNT\": 3, \"KAL\": 5, \"SUL\": 6, \"MAL\": 2, \"PAP\": 6}\n\ndef buat_data(seed=0):\n    rng = np.random.default_rng(seed)\n    rerata = {\"SUM\": (0.2, 0.0), \"JAW\": (0.8, 1.2), \"BNT\": (-0.6, -0.3),\n              \"KAL\": (0.4, 0.0), \"SUL\": (0.0, -0.3), \"MAL\": (-0.7, -0.4),\n              \"PAP\": (-1.8, -0.9)}\n    kode, F = [], []\n    for w, n in WILAYAH.items():\n        kode += [f\"{w}-{i+1:02d}\" for i in range(n)]\n        F.append(rng.normal(rerata[w], 0.45, (n, 2)))\n    F = np.vstack(F)                                   # faktor laten (38 x 2)\n    # muatan: kolom 1 = pembangunan, kolom 2 = urbanisasi\n    W = np.array([[-0.9, 0.0], [0.95, 0.2], [0.8, 0.4], [0.8, 0.0], [0.0, 0.8],\n                  [0.1, 0.6], [0.8, 0.3], [0.6, 0.7], [0.5, 0.6], [0.3, 0.9],\n                  [-0.8, -0.1]])\n    psi = np.array([.3, .15, .3, .4, .5, .7, .4, .3, .6, .25, .45])  # derau unik\n    Z = F @ W.T + rng.normal(0, 1, (38, 11)) * psi\n    skala = np.array([[10, 4], [71, 4], [8.6, 1.1], [70, 2], [5, 1.5], [0.37, 0.03],\n                      [88, 6], [70, 12], [3.8, 0.5], [50, 15], [22, 5]])\n    X = skala[:, 0] + Z * skala[:, 1]\n    return np.array(kode), X\n",
"bab12/ica_getaran.py": "import numpy as np\nfrom sklearn.decomposition import FastICA, PCA\n\n# Dua mesin di satu ruang (ilustrasi): press berdenyut, pompa bergetar gigi gergaji.\nt = np.linspace(0, 1, 2000)\ns1 = np.sign(np.sin(2 * np.pi * 7 * t))                       # denyut press\ns2 = 2 * ((5 * t) % 1) - 1                                    # gergaji pompa\nS = np.c_[s1, s2] + 0.05 * np.random.default_rng(0).normal(size=(2000, 2))\nA = np.array([[1.0, 0.6], [0.5, 1.0]])                        # pencampuran  #<1>\nX = S @ A.T                                                   # dua sensor\n\ndef korelasi_terbaik(Y):                                      # cocokkan sumber  #<2>\n    C = np.abs(np.corrcoef(S.T, Y.T)[:2, 2:])\n    return C.max(1)\n\nY_pca = PCA(2).fit_transform(X)\nica = FastICA(2, random_state=0, whiten=\"unit-variance\")      #<3>\nY_ica = ica.fit_transform(X)\nprint(\"|korelasi| dengan sumber asli (sumber 1, sumber 2):\")\nprint(\"  sensor mentah:\", np.round(korelasi_terbaik(X), 3))\nprint(\"  PCA          :\", np.round(korelasi_terbaik(Y_pca), 3))\nprint(\"  ICA          :\", np.round(korelasi_terbaik(Y_ica), 3))\nkurt = lambda y: np.mean(((y - y.mean()) / y.std()) ** 4) - 3  # non-Gaussian  #<4>\nprint(\"kurtosis lebih: sumber\", np.round([kurt(s1), kurt(s2)], 2),\n      \"| campuran\", np.round([kurt(X[:, 0]), kurt(X[:, 1])], 2))\n",
"bab12/nonlinear.py": "import time\nimport numpy as np\nfrom sklearn.datasets import make_circles\nfrom sklearn.decomposition import PCA, KernelPCA\nfrom sklearn.manifold import TSNE, trustworthiness\nfrom sklearn.linear_model import LogisticRegression\n\n# Bagian 1: dua cincin konsentris (ilustrasi), tidak terpisah secara linear.\nX, y = make_circles(400, factor=0.4, noise=0.06, random_state=0)\nakur = lambda Y: LogisticRegression().fit(Y, y).score(Y, y)   # pemisah linear\nprint(f\"PCA, 2 komponen       : akurasi {akur(PCA(2).fit_transform(X)):.2f}\")\nK = KernelPCA(3, kernel=\"rbf\", gamma=2.0).fit_transform(X)       #<1>\nfor k in [1, 2, 3]:\n    print(f\"kernel PCA, {k} komponen: akurasi {akur(K[:, :k]):.2f}\")\n\n# Bagian 2: 4 kelompok di permukaan melengkung, ditanam di 20 dimensi.\nrng = np.random.default_rng(0)\nt = rng.uniform(1.5 * np.pi, 4.5 * np.pi, 1200); h = rng.uniform(0, 10, 1200)\ng = np.digitize(t, np.linspace(1.5 * np.pi, 4.5 * np.pi, 5)[1:-1])   # 4 kelompok\nR3 = np.c_[t * np.cos(t), h, t * np.sin(t)]                         # gulungan  #<2>\nQ, _ = np.linalg.qr(rng.normal(size=(20, 20)))\nX20 = np.c_[R3, np.zeros((1200, 17))] @ Q + rng.normal(0, 0.3, (1200, 20))\nhasil = {}\nfor nama, m in [(\"PCA  \", PCA(2)), (\"t-SNE\", TSNE(2, perplexity=30, random_state=0))]:\n    t0 = time.perf_counter(); Y = m.fit_transform(X20); dt = time.perf_counter() - t0\n    tw = trustworthiness(X20, Y, n_neighbors=10)                    #<3>\n    print(f\"{nama}: trustworthiness 10-tetangga {tw:.3f}, waktu {dt:5.2f} s\")\n    hasil[nama.strip()] = Y\n",
"bab12/pca_provinsi.py": "import numpy as np\nfrom data_provinsi import buat_data, INDIKATOR\n\nkode, X = buat_data()\nZ = (X - X.mean(0)) / X.std(0)                       # baku: satuan berbeda-beda  #<1>\nS = Z.T @ Z / len(Z)                                 # = matriks korelasi\nlam, U = np.linalg.eigh(S)                           # nilai & vektor eigen  #<2>\nlam, U = lam[::-1], U[:, ::-1]                       # urutkan menurun\nU *= np.sign(U[1])                                   # tanda: IPM bermuatan positif\nporsi = lam / lam.sum()\nprint(\"porsi varians   :\", \" \".join(f\"{p:.2f}\" for p in porsi[:5]))\nprint(\"kumulatif       :\", \" \".join(f\"{p:.2f}\" for p in np.cumsum(porsi)[:5]))\nprint(\"muatan KU1, KU2 :\")\nfor nama, u1, u2 in zip(INDIKATOR, U[:, 0], U[:, 1]):\n    print(f\"   {nama:13s} {u1:+.2f} {u2:+.2f}\")\nY = Z @ U[:, :2]                                     # proyeksi ke 2 komponen  #<3>\nG = Y @ U[:, :2].T                                   # rekonstruksi\nprint(f\"galat rekonstruksi rata-rata: {np.mean((Z - G) ** 2):.3f}\"\n      f\" (= jumlah eigen sisanya / D: {lam[2:].sum() / len(lam):.3f})\")\n_, s, Vt = np.linalg.svd(Z, full_matrices=False)     # jalur SVD  #<4>\nprint(\"eigen dari SVD sama:\", np.allclose(s ** 2 / len(Z), lam))\nfor i in np.argsort(Y[:, 0])[[0, 1, -2, -1]]:\n    print(f\"   {kode[i]}: KU1 {Y[i, 0]:+.2f}, KU2 {Y[i, 1]:+.2f}\")\n",
"bab12/ppca_fa.py": "import numpy as np\nfrom scipy.stats import multivariate_normal as mvn\nfrom sklearn.decomposition import PCA, FactorAnalysis\nfrom sklearn.model_selection import cross_val_score\nfrom data_provinsi import buat_data, INDIKATOR\n\nkode, X = buat_data()\nZ = (X - X.mean(0)) / X.std(0)\nN, D = Z.shape\nlam, U = np.linalg.eigh(Z.T @ Z / N); lam, U = lam[::-1], U[:, ::-1]\n\ndef ppca(M):                                         # solusi bentuk tertutup  #<1>\n    s2 = lam[M:].mean()                              # rata-rata eigen yang dibuang\n    W = U[:, :M] * np.sqrt(lam[:M] - s2)\n    return W, s2\n\nW, s2 = ppca(2)\nC = W @ W.T + s2 * np.eye(D)                         # kovarians marginal  #<2>\nll = mvn.logpdf(Z, np.zeros(D), C).mean()\nprint(f\"PPCA M=2: sigma^2 = {s2:.3f}, log-lik./provinsi = {ll:.3f}\")\nprint(f\"  PCA(2).score di scikit-learn: {PCA(2).fit(Z).score(Z):.3f}\")\nfa = FactorAnalysis(2, random_state=0).fit(Z)        # derau berbeda per variabel  #<3>\nprint(\"varians derau FA  :\", \" \".join(f\"{v:.2f}\" for v in fa.noise_variance_))\nprint(f\"  terbesar: {INDIKATOR[fa.noise_variance_.argmax()]},\"\n      f\" terkecil: {INDIKATOR[fa.noise_variance_.argmin()]}\")\nprint(\" M | log-lik. validasi silang PPCA |  FA\")       #<4>\nfor M in range(1, 6):\n    a = cross_val_score(PCA(M), Z, cv=5).mean()\n    b = cross_val_score(FactorAnalysis(M, random_state=0), Z, cv=5).mean()\n    print(f\" {M} | {a:29.2f} | {b:5.2f}\")\n"
}
akar = pathlib.Path('pola-ml/kode').resolve()
for nama, isi in BERKAS.items():
    p = akar / nama; p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(isi, encoding='utf-8')
os.chdir(akar / 'bab12')
print('folder kerja:', os.getcwd())

## Listing 12.1: PCA dari nol pada 38 provinsi sintetis

Berkas: `kode/bab12/pca_provinsi.py`

In [ ]:
import numpy as np
from data_provinsi import buat_data, INDIKATOR

kode, X = buat_data()
Z = (X - X.mean(0)) / X.std(0)                       # baku: satuan berbeda-beda  # (1)
S = Z.T @ Z / len(Z)                                 # = matriks korelasi
lam, U = np.linalg.eigh(S)                           # nilai & vektor eigen  # (2)
lam, U = lam[::-1], U[:, ::-1]                       # urutkan menurun
U *= np.sign(U[1])                                   # tanda: IPM bermuatan positif
porsi = lam / lam.sum()
print("porsi varians   :", " ".join(f"{p:.2f}" for p in porsi[:5]))
print("kumulatif       :", " ".join(f"{p:.2f}" for p in np.cumsum(porsi)[:5]))
print("muatan KU1, KU2 :")
for nama, u1, u2 in zip(INDIKATOR, U[:, 0], U[:, 1]):
    print(f"   {nama:13s} {u1:+.2f} {u2:+.2f}")
Y = Z @ U[:, :2]                                     # proyeksi ke 2 komponen  # (3)
G = Y @ U[:, :2].T                                   # rekonstruksi
print(f"galat rekonstruksi rata-rata: {np.mean((Z - G) ** 2):.3f}"
      f" (= jumlah eigen sisanya / D: {lam[2:].sum() / len(lam):.3f})")
_, s, Vt = np.linalg.svd(Z, full_matrices=False)     # jalur SVD  # (4)
print("eigen dari SVD sama:", np.allclose(s ** 2 / len(Z), lam))
for i in np.argsort(Y[:, 0])[[0, 1, -2, -1]]:
    print(f"   {kode[i]}: KU1 {Y[i, 0]:+.2f}, KU2 {Y[i, 1]:+.2f}")

## Listing 12.2: PPCA bentuk tertutup dan analisis faktor

Berkas: `kode/bab12/ppca_fa.py`

In [ ]:
import numpy as np
from scipy.stats import multivariate_normal as mvn
from sklearn.decomposition import PCA, FactorAnalysis
from sklearn.model_selection import cross_val_score
from data_provinsi import buat_data, INDIKATOR

kode, X = buat_data()
Z = (X - X.mean(0)) / X.std(0)
N, D = Z.shape
lam, U = np.linalg.eigh(Z.T @ Z / N); lam, U = lam[::-1], U[:, ::-1]

def ppca(M):                                         # solusi bentuk tertutup  # (1)
    s2 = lam[M:].mean()                              # rata-rata eigen yang dibuang
    W = U[:, :M] * np.sqrt(lam[:M] - s2)
    return W, s2

W, s2 = ppca(2)
C = W @ W.T + s2 * np.eye(D)                         # kovarians marginal  # (2)
ll = mvn.logpdf(Z, np.zeros(D), C).mean()
print(f"PPCA M=2: sigma^2 = {s2:.3f}, log-lik./provinsi = {ll:.3f}")
print(f"  PCA(2).score di scikit-learn: {PCA(2).fit(Z).score(Z):.3f}")
fa = FactorAnalysis(2, random_state=0).fit(Z)        # derau berbeda per variabel  # (3)
print("varians derau FA  :", " ".join(f"{v:.2f}" for v in fa.noise_variance_))
print(f"  terbesar: {INDIKATOR[fa.noise_variance_.argmax()]},"
      f" terkecil: {INDIKATOR[fa.noise_variance_.argmin()]}")
print(" M | log-lik. validasi silang PPCA |  FA")       # (4)
for M in range(1, 6):
    a = cross_val_score(PCA(M), Z, cv=5).mean()
    b = cross_val_score(FactorAnalysis(M, random_state=0), Z, cv=5).mean()
    print(f" {M} | {a:29.2f} | {b:5.2f}")

## Listing 12.3: Memisahkan dua sumber getaran dengan FastICA

Berkas: `kode/bab12/ica_getaran.py`

In [ ]:
import numpy as np
from sklearn.decomposition import FastICA, PCA

# Dua mesin di satu ruang (ilustrasi): press berdenyut, pompa bergetar gigi gergaji.
t = np.linspace(0, 1, 2000)
s1 = np.sign(np.sin(2 * np.pi * 7 * t))                       # denyut press
s2 = 2 * ((5 * t) % 1) - 1                                    # gergaji pompa
S = np.c_[s1, s2] + 0.05 * np.random.default_rng(0).normal(size=(2000, 2))
A = np.array([[1.0, 0.6], [0.5, 1.0]])                        # pencampuran  # (1)
X = S @ A.T                                                   # dua sensor

def korelasi_terbaik(Y):                                      # cocokkan sumber  # (2)
    C = np.abs(np.corrcoef(S.T, Y.T)[:2, 2:])
    return C.max(1)

Y_pca = PCA(2).fit_transform(X)
ica = FastICA(2, random_state=0, whiten="unit-variance")      # (3)
Y_ica = ica.fit_transform(X)
print("|korelasi| dengan sumber asli (sumber 1, sumber 2):")
print("  sensor mentah:", np.round(korelasi_terbaik(X), 3))
print("  PCA          :", np.round(korelasi_terbaik(Y_pca), 3))
print("  ICA          :", np.round(korelasi_terbaik(Y_ica), 3))
kurt = lambda y: np.mean(((y - y.mean()) / y.std()) ** 4) - 3  # non-Gaussian  # (4)
print("kurtosis lebih: sumber", np.round([kurt(s1), kurt(s2)], 2),
      "| campuran", np.round([kurt(X[:, 0]), kurt(X[:, 1])], 2))

## Listing 12.4: Kernel PCA pada dua cincin, dan t-SNE pada gulungan berdimensi 20

Berkas: `kode/bab12/nonlinear.py`

In [ ]:
import time
import numpy as np
from sklearn.datasets import make_circles
from sklearn.decomposition import PCA, KernelPCA
from sklearn.manifold import TSNE, trustworthiness
from sklearn.linear_model import LogisticRegression

# Bagian 1: dua cincin konsentris (ilustrasi), tidak terpisah secara linear.
X, y = make_circles(400, factor=0.4, noise=0.06, random_state=0)
akur = lambda Y: LogisticRegression().fit(Y, y).score(Y, y)   # pemisah linear
print(f"PCA, 2 komponen       : akurasi {akur(PCA(2).fit_transform(X)):.2f}")
K = KernelPCA(3, kernel="rbf", gamma=2.0).fit_transform(X)       # (1)
for k in [1, 2, 3]:
    print(f"kernel PCA, {k} komponen: akurasi {akur(K[:, :k]):.2f}")

# Bagian 2: 4 kelompok di permukaan melengkung, ditanam di 20 dimensi.
rng = np.random.default_rng(0)
t = rng.uniform(1.5 * np.pi, 4.5 * np.pi, 1200); h = rng.uniform(0, 10, 1200)
g = np.digitize(t, np.linspace(1.5 * np.pi, 4.5 * np.pi, 5)[1:-1])   # 4 kelompok
R3 = np.c_[t * np.cos(t), h, t * np.sin(t)]                         # gulungan  # (2)
Q, _ = np.linalg.qr(rng.normal(size=(20, 20)))
X20 = np.c_[R3, np.zeros((1200, 17))] @ Q + rng.normal(0, 0.3, (1200, 20))
hasil = {}
for nama, m in [("PCA  ", PCA(2)), ("t-SNE", TSNE(2, perplexity=30, random_state=0))]:
    t0 = time.perf_counter(); Y = m.fit_transform(X20); dt = time.perf_counter() - t0
    tw = trustworthiness(X20, Y, n_neighbors=10)                    # (3)
    print(f"{nama}: trustworthiness 10-tetangga {tw:.3f}, waktu {dt:5.2f} s")
    hasil[nama.strip()] = Y